# TODO
1) Capire se non fare la media degli embedding degli spettri raggruppando per inchikey (vedi ultime due risposte gemini)
2) **settare n° di componenti fisse di pca in modo che molecole e proteine abbiano stesso numero di componenti**
3) usare UMAP invece di LDA
4) sistemare sezione "struttura notebook"
- magari provo anche Support vector machine anche se è dell'altro corso
- Potrebbe aver senso rimuovere quelle molecole che hanno dev std troppo alta (se > 15 se ne toglierebbero 12 su 714)
- fare una dimensionality reduction manuale non ha tanto senso perchè le var in input sono embedding senza quindi un significato chiaro

# Predizione della Bioattività di Molecole Prodotte dalle Piante

## Obiettivo
>Sviluppare modelli di machine learning per predire la **bioattività** di molecole prodotte dalle piante.

## Descrizione dataset
Il dataset è composto da 3 componenti:

### 1) "bioactivities" 
>Contiene l'elenco di bioattività tra molecole e proteine target. (source: [Papyrus](https://pmc.ncbi.nlm.nih.gov/articles/PMC9824924/))

- **Identificatori**
  - `Activity_ID` (*"primary key"*): identifica la **bioattività** di una coppia "molecola-proteina"
    - `InChIKey` (*"foreign key"*): identifica una specifica **molecola**
    - `target_id` (*"foreign key"*): identifica una specifica **proteina**

- **Statistiche bioattività**
  - `pchembl_value`: valori di bioattività derivanti da multiple misurazioni
    - 2 interpretazioni euristiche possibili:
      1) la più comune:
         - **Active**$~~~$: $\text{pChembl}  > 5.5$
         - **Inactive**: $\text{pChembl} <= 5.5$
      2) alternativa:
         - **Active**: $\text{pChembl} >= 6.0$
         - **Borderline**: $5.0 <= \text{pChembl} < 6.0$
         - **Inactive**: $\text{pChembl}  < 5.0$
  - `pchembl_value_Mean`: **media** dei valori di bioattività misurati
  - `pchembl_value_StdDev`: **deviazione standard** dei valori di bioattività misurati
  - `pchembl_value_SEM`: **errore standard della media** dei valori di bioattività misurati
  - `pchembl_value_N`: **numero** dei valori di bioattività misurati 
  - `pchembl_value_Median`: **mediana** dei valori di bioattività misurati
  - `pchembl_value_MAD`: **deviazione assoluta dalla mediana** dei valori di bioattività misurati

- **Metadati superflui**

	
### 2) "mass_spectra_embeddings"
>Contiene gli embedding delle molecole ottenuti con il tool python [Spec2Vec](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1008724) a partire dagli spettri di massa.

- **Identificatori**
  - `spectrum_id` (*"primary key"*): identifica uno specifico **spettro di una molecola**
    - `inchikey`: identifica una specifica **molecola**

- **Embedding features**
  - \[`0`,...,`499`\]: valori di embedding

- **Metadati superflui**

### 3) "protein_embeddings"
>Contiene 3 dimensionalità di embedding delle proteine ottenuti con UniRep. (source: [Papyrus](https://pmc.ncbi.nlm.nih.gov/articles/PMC9824924/))
1) File `protein_embeddings_UniRep64.csv`: embedding costituito da 3 componenti da 64 features/colonne, per un totale di 192 features
2) File `protein_embeddings_UniRep256.csv`: embedding costituito da 3 componenti da 256 features/colonne, per un totale di 768 features
3) File `protein_embeddings_UniRep1900.csv`: embedding costituito da 3 componenti da 1900 features/colonne, per un totale di 5700 features

- **Identificatori**
  - `target_id` (*"primary key"*): identifica una specifica **proteina**

- **Embedding features**
  - \[`UniRep64_AH_1`,...,`UniRep64_AH_64`,`UniRep64_FC_1`,...,`UniRep64_FC_64`,`UniRep64_FH_1`,...,`UniRep64_FH_64`\]: valori di embedding per ciascuna delle 3 componenti
  - \[`UniRep256_AH_1`,...,`UniRep256_AH_256`,`UniRep256_FC_256`,...
  - \[`UniRep1900_AH_1`,...,`UniRep1900_AH_1900`,`UniRep1900_FC_1900`,...


## Struttura del Notebook
1. Ipotesi e deduzioni
2. Configurazione ambiente ed import librerie
3. Caricamento dei dataset
4. Preprocessing
5. Training e validazione modelli
6. Salvataggio modelli
7. Conclusioni

---

# 1) Ipotesi e deduzioni
Dato che i dataset di molecole e proteine contengono rispettivamente
- 5798 embedding di spettri di massa per 714 molecole distinte (circa 8 per molecola) e
- 1142 embedding di proteine distinte

avere 10092 "bioattività" per 714 molecole e 1142 proteine significa che
- ciascuna **proteina** è stata testata con circa 9 **molecole** ($10092/1142=8.83$) e
- ciascuna **molecola** è stata testata con circa 14 **proteine** ($10092/714=14.13$)

Da ciò si evince che **si è cercato studiare il comportamento di "poche" molecole ma "a fondo"**.

Inoltre, la natura degli embedding di molecole e proteine è differente:
- embedding **molecole**: derivati da **spettri di massa**, misure fisiche reali soggette a rumore ambientale, strumentale, umano
- embedding **proteine**: derivati dalla **sequenza di amminoacidi** di ogni proteina che è quindi **fissa**

A fronte di ciò è stato scelto di **non aggregare** gli spettri di massa (mediandoli) in fase di join dei dataset in modo da permettere ai modelli di **distinguere** tra **rumore** e **informazione utile** a identificare la molecola.
___
Visto che l'obiettivo è costruire un classificatore che possa aiutare a identificare le molecole che hanno un "qualsiasi tipo di effetto" (bioattività) quando combinate con una proteina e che quindi vale la pena analizzare, la priorità è **evitare falsi negativi** perché i falsi positivi causeranno solo una piccola perdita di tempo in laboratorio.

Per questo motivo si è scelto di affrontare tale problema tramite **classificazione** piuttosto che **regressione** in modo da poter massimizzare il valore di ***recall*** invece di minimizzare l'errore sul valore di bioattività.

Inoltre, sono stati indicati 2 diversi criteri per classificare le bioattività e nessuna delle due dà origine a classi bilanciate infatti
- con soglia $5.5$:
  - 3651 `active`
  - 6441 `inactive`
- con soglie $5.0$ e $6.0$:
  - 2329 `active`
  - 6044 `borderline`
  - 1719 `inactive`

Per entrambi questi motivi è stato deciso di scegliere come unica soglia quella che **bilancia meglio le classi** cercando nell'intervallo compreso tra $5.0$ e $5.5$ inclusi: in questo modo c'è la possibilità di **bilanciare le classi** e al contempo di **sfavorire i falsi negativi** considerando come `active` anche alcuni valori `borderline`. 

Questo ha portato inoltre a scegliere la classificazione **binaria** rispetto a quella ternaria.
___
Riguardo la presenza di **3 diverse dimensionalità** per gli embedding delle proteine (UniRep 64, 256 e 1900), è stato scelto di utilizzare la **dimensionalità maggiore** (UniRep 1900). 

Questa decisione si basa sull'idea che sia **preferibile partire da una rappresentazione ricca di dettagli e sfumature strutturali, di cui estrarne l'essenza più informativa tramite algoritmi di riduzione dimensionale**.

Al contrario, **partire da embedding a bassa dimensionalità comporterebbe lavorare su dati già compressi a monte, con il rischio di aver perso irreversibilmente informazioni sottili ma cruciali che nessuna tecnica successiva potrebbe recuperare**.
___
Data la dimensionalità dei dataset, per la classificazione si è scelto di effettuare una feature reduction tramite
- **PCA** (tecnica adoperata dal modello di riferimento anche se non particolarmente indicata in ambito classificazione) e
- **LDA** (più adatto in ambito classificazione)

e di utilizzare i modelli
- **XGBoost** (tecnica adoperata dal modello di riferimento) e
- **Random forest** (perfetto per la classificazione)

In particolare verranno confrontate le combinazioni
- **PCA + XGBoost** (modello di riferimento)
- **LDA + XGBoost**
- **PCA + Random forest**
- **LDA + Random forest**
___

# 2) Configurazione ambiente ed import librerie

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.base import clone

# Models
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef
)

# Other
import os

# Set random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# 3) Caricamento ed esplorazione del dataset

In [ ]:
# Percorso base del dataset (modificare se necessario per Colab)
# Per Google Colab: montare prima il drive e modificare il path
#from google.colab import drive
#drive.mount('/content/drive')
#BASE_PATH = Path('/content/drive/MyDrive/Colab Notebooks/dataset_bioactivities')

os.makedirs('results', exist_ok=True) # Crea la cartella per l'output

# Per esecuzione locale
BASE_PATH = Path('dataset_bioactivities')

# Caricamento bioactivities
print("📊 Caricamento bioactivities...")
df_bioactivities = pd.read_csv(BASE_PATH / 'bioactivities' / 'dataset_bioactivities_negative.csv')

# Caricamento molecule embeddings
print("📊 Caricamento mass_spectra_embeddings...")
df_molecules = pd.read_csv(BASE_PATH / 'mass_spectra_embeddings' / 'ALL_GNPS_negative_ionization_DATASET.csv')

# Caricamento protein embeddings
print("📊 Caricamento protein_embeddings...")
df_proteins_1900 = pd.read_csv(BASE_PATH / 'protein_embeddings' / 'protein_embeddings_UniRep1900.csv')

print(f"\n✅ Dataset caricati con successo!")
print(f"   - Bioactivities: {df_bioactivities.shape}")
print(f"   - Molecules: {df_molecules.shape}")
print(f"   - Proteins (UniRep1900): {df_proteins_1900.shape}")

📊 Caricamento bioactivities...
📊 Caricamento mass_spectra_embeddings...
📊 Caricamento protein_embeddings...

✅ Dataset caricati con successo!
   - Bioactivities: (10092, 36)
   - Molecules: (5798, 507)
   - Proteins (UniRep1900): (1142, 5701)


In [33]:
df_bioactivities

,Activity_ID,Quality,source,CID,SMILES,connectivity,InChIKey,InChI,InChI_AuxInfo,target_id,...,pchembl_value_N,pchembl_value_Median,pchembl_value_MAD,HGNC_symbol,UniProtID,Status,Organism,Classification,Length,Sequence
0,AAKJLRGGTJKAMG_on_P00533_WT,High,ChEMBL30,CHEMBL553;2050;2050;2050;CHEMBL553;CHEMBL553;2...,C#Cc1cc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)ccc1,AAKJLRGGTJKAMG,AAKJLRGGTJKAMG-UHFFFAOYSA-N,InChI=1S/C22H23N3O4/c1-4-16-6-5-7-17(12-16)25-...,"""AuxInfo=1/1/N:1,24,18,2,28,29,27,22,16,21,15,...",P00533_WT,...,514,7.550,0.86,EGFR,EGFR_HUMAN,reviewed,Homo sapiens (Human),Enzyme->Kinase->Protein Kinase->TK protein kin...,1210,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...
1,ACPOUJIDANTYHO_on_P00533_WT,High,Sharma2016,1800164;Merget2017.compound.1660;Christmann201...,O=C1c2c(cccc2)-c2c3c1cccc3n[nH]2,ACPOUJIDANTYHO,ACPOUJIDANTYHO-UHFFFAOYSA-N,InChI=1S/C14H8N2O/c17-14-9-5-2-1-4-8(9)13-12-1...,"""AuxInfo=1/1/N:6,7,13,5,8,12,14,4,3,11,15,10,9...",P00533_WT,...,14,5.700,0.00,EGFR,EGFR_HUMAN,reviewed,Homo sapiens (Human),Enzyme->Kinase->Protein Kinase->TK protein kin...,1210,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...
2,AFSDNFLWKVMVRB_on_P00533_WT,High,ExCAPE-DB,5281855;Merget2017.compound.1659;CHEMBL6246;52...,O=c1oc2c3c(cc(O)c2O)c(=O)oc2c(O)c(O)cc1c23,AFSDNFLWKVMVRB,AFSDNFLWKVMVRB-UHFFFAOYSA-N,InChI=1S/C14H6O8/c15-5-1-3-7-8-4(14(20)22-11(7...,"""AuxInfo=1/0/N:7,20,6,21,8,18,5,22,10,16,4,15,...",P00533_WT,...,12,6.161,0.00,EGFR,EGFR_HUMAN,reviewed,Homo sapiens (Human),Enzyme->Kinase->Protein Kinase->TK protein kin...,1210,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...
3,AILRADAXUVEEIR_on_P00533_WT,High,ChEMBL30,CHEMBL3545311,COc1cc(N2CCC(N3CCN(C)CC3)CC2)ccc1Nc1ncc(Cl)c(N...,AILRADAXUVEEIR,AILRADAXUVEEIR-UHFFFAOYSA-N,InChI=1S/C29H39ClN7O2P/c1-35-15-17-37(18-16-35...,"""AuxInfo=1/1/N:14,1,37,38,32,33,31,34,19,20,8,...",P00533_WT,...,34,8.600,0.40,EGFR,EGFR_HUMAN,reviewed,Homo sapiens (Human),Enzyme->Kinase->Protein Kinase->TK protein kin...,1210,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...
4,ATEFPOUAMCWAQS_on_P00533_WT,High,Sharma2016,14348,O=c1ccc2c(o1)c(O)c(O)cc2,ATEFPOUAMCWAQS,ATEFPOUAMCWAQS-UHFFFAOYSA-N,InChI=1S/C9H6O4/c10-6-3-1-5-2-4-7(11)13-9(5)8(...,"""AuxInfo=1/0/N:13,4,12,3,5,10,2,8,6,11,1,9,7/r...",P00533_WT,...,8,5.115,0.00,EGFR,EGFR_HUMAN,reviewed,Homo sapiens (Human),Enzyme->Kinase->Protein Kinase->TK protein kin...,1210,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10087,XNYZHCFCZNMTFY_on_P19801_WT,High,ChEMBL30,CHEMBL35241,N=C(N)c1ccc(N=NNc2ccc(C(=N)N)cc2)cc1,XNYZHCFCZNMTFY,XNYZHCFCZNMTFY-UHFFFAOYSA-N,InChI=1S/C14H15N7/c15-13(16)9-1-5-11(6-2-9)19-...,"""AuxInfo=1/1/N:5,21,13,18,6,20,12,19,4,14,7,11...",P19801_WT,...,2,7.890,0.00,NaN,AOC1_HUMAN,reviewed,Homo sapiens (Human),Enzyme,751,MPALGWAVAAILMLQTAMAEPSPGTLPRKAGVFSDLSNQELKAVHS...
10088,XNYZHCFCZNMTFY_on_P21673_WT,High,ChEMBL30,CHEMBL35241,N=C(N)c1ccc(N=NNc2ccc(C(=N)N)cc2)cc1,XNYZHCFCZNMTFY,XNYZHCFCZNMTFY-UHFFFAOYSA-N,InChI=1S/C14H15N7/c15-13(16)9-1-5-11(6-2-9)19-...,"""AuxInfo=1/1/N:5,21,13,18,6,20,12,19,4,14,7,11...",P21673_WT,...,2,5.620,0.00,NaN,SAT1_HUMAN,reviewed,Homo sapiens (Human),Enzyme->Transferase,171,MAKFVIRPATAADCSDILRLIKELAKYEYMEEQVILTEKDLLEDGF...
10089,XUZMWHLSFXCVMG_on_Q01844_WT,High,Klaeger2017,Klaeger2017.compound.37,CCS(=O)(=O)N1CC(CC#N)(n2cc(-c3ncnc4c3cc[nH]4)c...,XUZMWHLSFXCVMG,XUZMWHLSFXCVMG-UHFFFAOYSA-N,"InChI=1S/C16H17N7O2S/c1-2-26(24,25)22-9-16(10-...","""AuxInfo=1/1/N:1,2,21,9,10,22,24,13,7,26,17,14...",Q01844_WT,...,4,7.745,0.00,EWSR1,EWS_HUMAN,reviewed,Homo sapiens (Human),Unclassified protein,656,MASTDYSTYSQAAAQQGYSAYTAQPTQGYAQTTQAYGQQSYGTYGQ...
10090,XUZMWHLSFXCVMG_on_Q9NY27_WT,High,Klaeger2017,Klaeger2017.compound.37,CCS(=O)(=O)N1CC(CC#N)(n2cc(-c3ncnc4c3cc[nH]4)c...,XUZMWHLSFXCVMG,XUZMWHLSFXCVMG-UHFFFAOYSA-N,"InChI=1S/C16H17N7O2S/c1-2-26(24,25)22-9-16(10-...","""AuxInfo=1/1/N:1,2,21,9,10,22,24,13,7,26,17,14...",Q9NY27_WT,...,4,7.25

In [34]:
df_molecules

,inchikey,backbone_inchikey,spectrum_id,smiles,inchi,instrument,ms_level,0,1,2,...,490,491,492,493,494,495,496,497,498,499
0,REFJWTPEDVJJIY-UHFFFAOYSA-N,REFJWTPEDVJJIY,CCMSLIB00000077171,C1=CC(=C(C=C1C2=C(C(=O)C3=C(C=C(C=C3O2)O)O)O)O)O,InChI=1S/C15H10O7/c16-7-4-10(19)12-11(5-7)22-1...,Ion Trap,2,9.607846,2.802023,8.878453,...,-2.267261,7.410372,-0.767007,-14.296808,12.043738,-0.535357,3.684344,-2.610004,4.438908,22.484255
1,DANYIYRPLHHOCZ-UHFFFAOYSA-N,DANYIYRPLHHOCZ,CCMSLIB00000077212,O=C1C2=C(C=C(C=C2OC(C3=CC=C(C=C3)OC)=C1)O)O,InChI=1S/C16H12O5/c1-20-11-4-2-9(3-5-11)14-8-1...,Ion Trap,2,-3.743248,8.963512,-13.119514,...,0.850199,1.896127,-1.428571,2.617292,0.967600,-7.100490,4.839757,2.050162,-2.505632,8.013733
2,REFJWTPEDVJJIY-UHFFFAOYSA-N,REFJWTPEDVJJIY,CCMSLIB00000077221,O=C1C2=C(C=C(C=C2OC(C3=CC(O)=C(C=C3)O)=C1O)O)O,InChI=1S/C15H10O7/c16-7-4-10(19)12-11(5-7)22-1...,Ion Trap,2,15.296115,5.382534,11.653003,...,-3.729640,7.611222,-0.269078,-31.249656,19.250404,-0.056575,8.586094,-11.137173,8.047441,27.927703
3,WEPBGSIAWZTEJR-UHFFFAOYSA-N,WEPBGSIAWZTEJR,CCMSLIB00000077223,O=C1C2=C(O)C=C(O)C=C2OC(C3=CC(O)=C(O)C=C3)=C1OC,InChI=1S/C16H12O7/c1-22-16-14(21)13-11(20)5-8(...,qTof,2,2.587487,-9.726512,-5.610098,...,13.326739,-1.829110,11.330718,2.624641,12.474549,-16.206268,-9.119176,-3.557700,-22.368920,-9.876165
4,FXNFHKRTJBSTCS-UHFFFAOYSA-N,FXNFHKRTJBSTCS,CCMSLIB00000077226,O=C1C2=C(O)C(O)=C(O)C=C2OC(C3=CC=CC=C3)=C1,InChI=1S/C15H10O5/c16-9-6-11(8-4-2-1-3-5-8)20-...,qTof,2,-11.276055,3.991606,-6.219286,...,20.143283,-12.907214,20.991170,6.716496,-12.201179,-23.204450,-18.559520,2.654958,-32.627998,18.730554
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5793,DANYIYRPLHHOCZ-UHFFFAOYSA-N,DANYIYRPLHHOCZ,CCMSLIB00004683981,COc1ccc(cc1)c2cc(=O)c3c(cc(cc3o2)O)O,InChI=1S/C16H12O5/c1-20-11-4-2-9(3-5-11)14-8-1...,impact HD,2,14.329692,13.079878,5.756769,...,6.417844,-2.512271,6.751256,1.389466,-3.304776,-1.515792,-13.919465,-6.961419,-9.796392,0.059967
5794,QIQXTHQIDYTFRH-UHFFFAOYSA-N,QIQXTHQIDYTFRH,CCMSLIB00004684075,CCCCCCCCCCCCCCCCCC(=O)O,InChI=1S/C18H36O2/c1-2-3-4-5-6-7-8-9-10-11-12-...,impact HD,2,6.989496,4.729643,-15.990228,...,3.185644,7.655814,14.964863,1.702161,-6.624478,-6.515879,-3.857400,13.081180,6.470927,-0.355202
5795,LKOJGSWUMISDOF-UHFFFAOYSA-N,LKOJGSWUMISDOF,CCMSLIB00004684099,COc1c(cc2c(c1O)c(=O)cc(o2)c3ccccc3)O,InChI=1S/C16H12O5/c1-20-16-11(18)8-13-14(15(16...,impact HD,2,1.996257,-3.806559,-7.551622,...,7.084883,-1.750906,8.428929,-3.933588,12.208838,-5.254452,-10.687805,-2.370923,-13.982405,-10.424351
5796,LKOJGSWUMISDOF-UHFFFAOYSA-N,LKOJGSWUMISDOF,CCMSLIB00004684100,COc1c(cc2c(c1O)c(=O)cc(o2)c3ccccc3)O,InChI=1S/C16H12O5/c1-20-16-11(18)8-13-14(15(16...,impact HD,2,11.374275,-5.898643,-5.371347,...,-7.863005,-6.672494,13.551168,14.498432,18.518915,-12.364852,-37.476517,0.812095,-26.493843,-13.222843


In [35]:
df_proteins_1900

,target_id,UniRep1900_AH_1,UniRep1900_AH_2,UniRep1900_AH_3,UniRep1900_AH_4,UniRep1900_AH_5,UniRep1900_AH_6,UniRep1900_AH_7,UniRep1900_AH_8,UniRep1900_AH_9,...,UniRep1900_FH_1891,UniRep1900_FH_1892,UniRep1900_FH_1893,UniRep1900_FH_1894,UniRep1900_FH_1895,UniRep1900_FH_1896,UniRep1900_FH_1897,UniRep1900_FH_1898,UniRep1900_FH_1899,UniRep1900_FH_1900
0,P08908_WT,0.08561,-0.03444,0.06463,-0.00829,-0.21623,0.12827,-0.06974,-0.10380,-0.00338,...,0.00427,-0.02972,0.01276,0.17572,-0.87879,0.07743,-0.04048,0.03391,0.24940,-0.31621
1,P78536_WT,0.00465,-0.14043,0.06790,-0.00596,-0.06450,0.01991,-0.16443,-0.01685,-0.00337,...,0.01954,-0.01483,-0.02285,0.04948,-0.01315,-0.13613,-0.00511,-0.01048,0.49837,0.02871
2,P00533_WT,0.00785,-0.20279,0.07403,-0.01518,-0.21333,-0.00485,-0.16621,-0.04478,-0.00409,...,0.01549,-0.00849,-0.05154,0.00772,-0.04419,-0.12872,0.24357,0.00958,0.26656,0.10337
3,P04626_WT,0.00854,-0.19614,0.05564,-0.01072,-0.18002,-0.00089,-0.19169,-0.04413,-0.00429,...,0.01564,0.00684,-0.02390,0.01311,-0.13924,-0.03414,0.00464,0.00206,0.36447,0.09659
4,P06239_WT,0.01757,-0.05984,0.05179,-0.00781,-0.14472,-0.01622,-0.14839,-0.07724,-0.00462,...,0.23025,-0.00522,0.01709,0.00819,0.22711,-0.31987,-0.03916,0.03440,0.45563,-0.28214
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1137,P10721_L576P,0.00900,-0.17074,0.11174,-0.00584,-0.39371,-0.02549,-0.12715,-0.02510,-0.00365,...,0.00634,0.08509,-0.17583,0.01217,-0.20080,-0.19077,-0.02889,0.04159,0.10743,0.19135
1138,P36888_R834Q,0.00730,-0.14832,0.09050,-0.00550,-0.38044,-0.01511,-0.11862,-0.03289,-0.00366,...,0.01012,0.01746,-0.01968,0.05201,-0.05188,-0.12221,-0.21007,0.00282,0.36529,0.03123
1139,P10721_A829P,0.00857,-0.17122,0.11295,-0.00626,-0.39561,-0.02463,-0.12432,-0.02574,-0.00366,...,0.00656,0.08659,-0.17109,0.01218,-0.19862,-0.19259,-0.03026,0.04174,0.11557,0.17989
1140,P08581_Y1235D,0.00641,-0.08644,0.09460,-0.00912,-0.28713,-0.00243,-0.12211,-0.03658,-0.00314,...,0.01419,0.05083,-0.01891,0.03963,-0.05231,-0.12792,0.06560,0.01853,0.53778,-0.01641


# 4) Preprocessing

## Data cleaning
La parte di Data Cleaning è già stata effettuata infatti:
- non ci sono dati mancanti (a parte nei metadati superflui)
- nel dataset delle bioattività 
  - il dominio della feature `Quality` contiene solo la qualità "**High**"
  - il dominio della feature `Status` contiene solo il valore **reviewed**
    - ogni misurazione è stata controllata manualmente
- il dominio della feature `ms_level` del dataset delle molecole contiene solo il valore $2$
  - gli embedding sono ricchi di informazione perché comprendono le singole componenti della molecola invece che solo la molecola intera (`ms_level` $= 1$)

## Feature selection

>Rimozione **metadati superflui** e **statistiche di bioattività** superflue.

In [36]:
# come statistica di bioattività si prende come riferimento pchembl_value_Mean
df_bioactivities.drop(columns=[
    'Quality', 'source', 'CID', 'SMILES', 'connectivity', 'InChI', 'InChI_AuxInfo', 'accession',
    'Protein_Type', 'AID', 'doc_id', 'Year', 'all_doc_ids', 'all_years', 'type_IC50', 'type_EC50',
    'type_KD', 'type_Ki', 'relation', 'HGNC_symbol', 'UniProtID', 'Status', 'Organism', 'Classification',
    'Length', 'Sequence',
    'pchembl_value', 'pchembl_value_StdDev', 'pchembl_value_SEM',
    'pchembl_value_N', 'pchembl_value_Median', 'pchembl_value_MAD'    
], inplace=True)

df_molecules.drop(columns=[
    'backbone_inchikey', 'spectrum_id', 'smiles', 'inchi', 'instrument', 'ms_level'
], inplace=True)

## Data preparation

>Si cerca la soglia che meglio bilancia le classi.

In [37]:
# Si ricerca nel range "safe" delle soglie (5.00-5.50)
pchembl_values = np.array(df_bioactivities['pchembl_value_Mean'].values)
thresholds = np.arange(5.00, 5.51, 0.01)

# Si contano gli attivi per ogni soglia (e gli inattivi per differenza)
active_counts = np.array([(pchembl_values >= t).sum() for t in thresholds])
inactive_counts = len(pchembl_values) - active_counts

# Si individua la soglia che minimizza la differenza tra le classi
best_idx = np.argmin(np.abs(active_counts - inactive_counts))
best_balancing_threshold = thresholds[best_idx]

print(f"Soglia ottimale: {best_balancing_threshold:.2f}")
print(f"active: {active_counts[best_idx]} | inactive: {inactive_counts[best_idx]}")

Soglia ottimale: 5.16
active: 5058 | inactive: 5034


>Viene aggiunta la colonna indicante la classe per ogni bioattività secondo la soglia trovata.

In [38]:
df_bioactivities['bioactivity_class'] = np.where(
    df_bioactivities['pchembl_value_Mean'] >= best_balancing_threshold,
    'active',
    'inactive'
)

>Si effettua il merge tra i data set **verificando che le classi rimangano bilanciate** anche dopo l'aumento di record dovuto alla presenza di più spettri per singola molecola.

In [39]:
# Rinomino la colonna InChIKey per uniformarla
df_bioactivities.rename(columns={'InChIKey': 'inchikey'}, inplace=True)

# Merge tra dataset bioattività e spettri molecole
df_merged = df_bioactivities.merge(df_molecules, on='inchikey', how='inner')

# Merge tra bioattività e spettri molecole con embedding proteine
df_merged = df_merged.merge(df_proteins_1900, on='target_id', how='inner')

print(f"\n✅ Dataset merged:")
print(f"df_merged: {df_merged.shape}\n")

# Calcola proporzione originale
prop_orig = df_bioactivities['bioactivity_class'].value_counts(normalize=True)

# Calcola nuova proporzione
prop_new = df_merged['bioactivity_class'].value_counts(normalize=True)

print("--- BILANCIAMENTO CLASSI ---")
print(f"Originale (Active): {prop_orig.get('active', 0):.2%}")
print(f"Merged    (Active): {prop_new.get('active', 0):.2%}")


✅ Dataset merged:
df_merged: (106055, 6205)

--- n° RECORD ---
Merged: 10092
Merged: 106055 (10092x1.1e+01)

--- BILANCIAMENTO CLASSI ---
Originale (Active): 50.12%
Merged    (Active): 53.38%


## Split con StratifiedGroupKFold (k=5)

Si effettua una **Cross-Validation con k=5 fold** usando `StratifiedGroupKFold` per:
1. **Stratificazione**: mantenere la distribuzione di `active`/`inactive` in ogni fold
2. **Raggruppamento per proteina**: in questo modo si evita che il modello impari la coppia proteina-bioattività visto che alcune proteine compaiono come **sempre** `active` o **sempre** `inactive`

> **Nota**: Standardizzazione e riduzione dimensionalità vengono applicate **dentro il loop di CV** (fit su train, transform su entrambi) per evitare data leakage.

In [ ]:
# Colonne non-feature
non_feature_cols = ['Activity_ID', 'inchikey', 'target_id', 'pchembl_value_Mean', 'bioactivity_class']

# Identifico colonne feature
feature_cols = [c for c in df_merged.columns if c not in non_feature_cols]

# Identifico indici delle colonne molecola (numeriche: '0'-'499') e proteina (UniRep*)
mol_idx = [i for i, c in enumerate(feature_cols) if c.isdigit() or (c[0] == '-' and c[1:].isdigit())]
prot_idx = [i for i, c in enumerate(feature_cols) if c.startswith('UniRep')]

# Preparo i dati per la CV
X = df_merged[feature_cols].values
y = df_merged['bioactivity_class'].values
groups = df_merged['target_id'].values

print(f"📊 Dataset preparato:")
print(f"   - {len(mol_idx)} features molecola + {len(prot_idx)} features proteina = {len(feature_cols)} totali")
print(f"   - {len(np.unique(groups))} proteine distinte")
print(f"   - {len(X)} campioni totali")

# Configurazione CV
N_SPLITS = 5
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
print(f"\n✅ StratifiedGroupKFold con {N_SPLITS} fold configurato")

64: 500 mol features + 192 prot features = 692 total, 714 molecole
256: 500 mol features + 768 prot features = 1268 total, 714 molecole
1900: 500 mol features + 5700 prot features = 6200 total, 714 molecole
64_no_reduction: 500 mol features + 192 prot features = 692 total, 714 molecole

✅ StratifiedGroupKFold con 5 fold configurato


# 5) Training e Validazione con Cross-Validation

Si addestrano e validano i modelli **XGBoost** e **Random Forest** con le seguenti tecniche di riduzione dimensionalità:
- **PCA** (a 32, 64, 128, 256, 500 componenti, applicato separatamente su molecole e proteine)
- **LDA** (1 componente)

**Metriche calcolate**: Accuracy, Precision, Recall, F1-score, AUC, MCC

In [ ]:
# Setup - Mappatura manuale delle classi (active=1, inactive=0)
def encode_y(y_raw):
    """Converte le label: active → 1, inactive → 0"""
    return (y_raw == 'active').astype(int)

# Encode delle label
y_enc = encode_y(y)
print(f"Mappatura classi: inactive=0, active=1")

# CPU assegnate con fallback a 8
n_jobs = int(os.environ.get('SLURM_CPUS_PER_TASK', 8))

# Definizione modelli
models = {
    'XGBoost': xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', device='cuda', n_jobs=n_jobs),
    'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100, n_jobs=n_jobs)
}

# Configurazioni PCA: numero fisso di componenti per ciascun blocco (molecole e proteine)
PCA_N_COMPONENTS = [32, 64, 128, 256, 500]

all_results = defaultdict(lambda: defaultdict(list))

def evaluate_fold(X_train, X_val, y_train, y_val, model, key):
    """Addestra e valuta un modello su un fold. pos_label=1 (active) è il default."""

    # Clona il modello base (così ogni fold parte da zero)
    clf = clone(model)

    # 1. TRAIN: Addestriamo usando la configurazione globale (GPU se disponibile)
    clf.fit(X_train, y_train)

    # 2. FIX WARNING: Se il modello è XGBoost e sta usando la GPU,
    # lo spostiamo su CPU solo per la fase di predizione.
    # Questo evita il conflitto "Input su CPU vs Modello su GPU".
    if 'XGBClassifier' in str(type(clf)) and clf.device == 'cuda':
        clf.set_params(device='cpu')
    
    # 3. PREDICT: Ora avviene senza warning
    y_pred = clf.predict(X_val)
    y_proba = clf.predict_proba(X_val)[:, 1]  # Probabilità classe 1 (active)
    
    # Calcolo metriche
    all_results[key]['Accuracy'].append(accuracy_score(y_val, y_pred))
    all_results[key]['Precision'].append(precision_score(y_val, y_pred))  # pos_label=1 default
    all_results[key]['Recall'].append(recall_score(y_val, y_pred))
    all_results[key]['F1-score'].append(f1_score(y_val, y_pred))
    all_results[key]['AUC'].append(roc_auc_score(y_val, y_proba))
    all_results[key]['MCC'].append(matthews_corrcoef(y_val, y_pred))

def apply_pca_separated(X_tr, X_val, n_components):
    """Applica PCA separato su molecole e proteine con scaling."""
    scaler_mol, scaler_prot = StandardScaler(), StandardScaler()
    # n_components viene limitato al massimo numero di feature del blocco
    n_mol = min(n_components, len(mol_idx))
    n_prot = min(n_components, len(prot_idx))
    pca_mol = PCA(n_components=n_mol, random_state=RANDOM_STATE)
    pca_prot = PCA(n_components=n_prot, random_state=RANDOM_STATE)
    
    X_tr_mol = pca_mol.fit_transform(scaler_mol.fit_transform(X_tr[:, mol_idx]))
    X_tr_prot = pca_prot.fit_transform(scaler_prot.fit_transform(X_tr[:, prot_idx]))
    X_val_mol = pca_mol.transform(scaler_mol.transform(X_val[:, mol_idx]))
    X_val_prot = pca_prot.transform(scaler_prot.transform(X_val[:, prot_idx]))
    
    scaler_final = StandardScaler()
    return scaler_final.fit_transform(np.hstack([X_tr_mol, X_tr_prot])), \
           scaler_final.transform(np.hstack([X_val_mol, X_val_prot]))

def apply_lda(X_tr, X_val, y_tr):
    """Applica LDA su tutte le feature."""
    scaler = StandardScaler()
    lda = LDA(n_components=1)
    X_train_scaled = scaler.fit_transform(X_tr)
    return lda.fit_transform(X_train_scaled, y_tr), lda.transform(scaler.transform(X_val))

# Training con Cross-Validation
print("\n" + "=" * 75)
print("🚀 TRAINING E VALIDAZIONE")
print("=" * 75)

# Lista per salvare le statistiche
fold_stats = []

for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X, y_enc, groups)):
    X_tr, X_val_raw = X[train_idx], X[val_idx]
    y_tr, y_val = y_enc[train_idx], y_enc[val_idx]

    # Calcolo statistiche del fold corrente
    train_size = len(train_idx)
    val_size = len(val_idx)
    train_active_perc = (y_tr == 1).sum() / train_size * 100
    val_active_perc = (y_val == 1).sum() / val_size * 100
    fold_stats.append((fold_idx + 1, train_size, val_size, train_active_perc, val_active_perc))
    
    # PCA con diversi numeri di componenti
    for n_comp in PCA_N_COMPONENTS:
        X_train_red, X_val_red = apply_pca_separated(X_tr, X_val_raw, n_comp)
        for model_name, model in models.items():
            key = f'pca{n_comp}_{model_name}'
            evaluate_fold(X_train_red, X_val_red, y_tr, y_val, model, key)
    
    # LDA
    X_train_red, X_val_red = apply_lda(X_tr, X_val_raw, y_tr)
    for model_name, model in models.items():
        key = f'lda_{model_name}'
        evaluate_fold(X_train_red, X_val_red, y_tr, y_val, model, key)

# Stampa risultati per ogni configurazione
print("\n📈 Risultati per configurazione:")
for key in all_results:
    recall = all_results[key]['Recall']
    print(f"   {key}: Recall = {np.mean(recall):.3f} ± {np.std(recall):.3f}")

# Tabella risultati finale
results_summary = []
for key, metrics in all_results.items():
    # Parsing chiave: 'pca32_XGBoost' → ('PCA(32)', 'XGBoost'), 'lda_RandomForest' → ('LDA', 'RandomForest')
    if key.startswith('pca'):
        underscore_pos = key.index('_')
        reduction = f"PCA({key[3:underscore_pos]})"
        model_part = key[underscore_pos+1:]
    else:
        parts = key.split('_')
        reduction = parts[0].upper()
        model_part = parts[1]
    results_summary.append({
        'Reduction': reduction,
        'Model': model_part,
        'Accuracy': f"{np.mean(metrics['Accuracy']):.3f}±{np.std(metrics['Accuracy']):.3f}",
        'Precision': f"{np.mean(metrics['Precision']):.3f}±{np.std(metrics['Precision']):.3f}",
        'Recall': f"{np.mean(metrics['Recall']):.3f}±{np.std(metrics['Recall']):.3f}",
        'F1-score': f"{np.mean(metrics['F1-score']):.3f}±{np.std(metrics['F1-score']):.3f}",
        'MCC': f"{np.mean(metrics['MCC']):.3f}±{np.std(metrics['MCC']):.3f}",
        '_recall': np.mean(metrics['Recall'])
    })

df_results = pd.DataFrame(results_summary).sort_values('_recall', ascending=False).drop(columns=['_recall'])
print("\n📊 TABELLA RISULTATI FINALE:")
print(df_results)

# Stampa della tabella dei fold
print("\n" + "="*80)
print("🧐 DETTAGLIO DISTRIBUZIONE NEI FOLD (Training Recap)")
print("="*80)
print(f"{'Fold':^5} | {'Train Size':^10} | {'Val Size':^10} | {'Train Active %':^15} | {'Val Active %':^15}")
print("-" * 75)
for row in fold_stats:
    print(f"{row[0]:^5} | {row[1]:^10} | {row[2]:^10} | {row[3]:^15.2f}% | {row[4]:^15.2f}%")
print("-" * 75)

Mappatura classi: inactive=0, active=1
64_pca_XGBoost: Recall=0.712±0.048
64_pca_RandomForest: Recall=0.733±0.045
64_lda_XGBoost: Recall=0.557±0.134
64_lda_RandomForest: Recall=0.546±0.125
256_pca_XGBoost: Recall=0.720±0.053
256_pca_RandomForest: Recall=0.766±0.047
256_lda_XGBoost: Recall=0.532±0.123
256_lda_RandomForest: Recall=0.535±0.105
1900_pca_XGBoost: Recall=0.737±0.029
1900_pca_RandomForest: Recall=0.809±0.031
1900_lda_XGBoost: Recall=0.565±0.144
1900_lda_RandomForest: Recall=0.570±0.136
64_no_reduction_XGBoost: Recall=0.730±0.034
64_no_reduction_RandomForest: Recall=0.779±0.031


,Dataset,Model,Accuracy,Precision,Recall,F1-score,AUC
9,1900_pca,RandomForest,0.582±0.062,0.565±0.096,0.809±0.031,0.660±0.073,0.625±0.011
13,64_no_reduction,RandomForest,0.579±0.060,0.568±0.098,0.779±0.031,0.650±0.068,0.633±0.032
5,256_pca,RandomForest,0.587±0.031,0.569±0.081,0.766±0.047,0.649±0.062,0.624±0.023
8,1900_pca,XGBoost,0.590±0.032,0.576±0.084,0.737±0.029,0.643±0.058,0.626±0.026
1,64_pca,RandomForest,0.585±0.033,0.578±0.090,0.733±0.045,0.639±0.049,0.620±0.026
12,64_no_reduction,XGBoost,0.601±0.040,0.588±0.097,0.730±0.034,0.647±0.067,0.640±0.031
4,256_pca,XGBoost,0.599±0.034,0.584±0.090,0.720±0.053,0.642±0.069,0.641±0.028
0,64_pca,XGBoost,0.573±0.038,0.568±0.092,0.712±0.048,0.625±0.058,0.619±0.029
11,1900_lda,RandomForest,0.527±0.072,0.548±0.126,0.570±0.136,0.542±0.106,0.538±0.063
10,1900_lda,XGBoost,0.524±0.077,0.543±0.126,0.565±0.144,0.536±0.115,0.549±0.059


# 6) Salvataggio modelli

In [ ]:
# import joblib
# os.makedirs('models', exist_ok=True)
#
# trained_models = {}
#
# # Ri-addestramento su tutto il dataset
# X_full = df_merged[feature_cols].values
# y_full = encode_y(df_merged['bioactivity_class'].values)
#
# # PCA con diversi numeri di componenti
# for n_comp in PCA_N_COMPONENTS:
#     X_red = apply_pca_separated(X_full, X_full, n_comp)[0]  # solo il primo output (fit su tutto)
#     for model_name, model in models.items():
#         clf = clone(model).fit(X_red, y_full)
#         key = f'pca{n_comp}_{model_name}'
#         trained_models[key] = clf
#         joblib.dump(clf, f'models/{key}.joblib')
#
# # LDA
# scaler = StandardScaler()
# lda = LDA(n_components=1)
# X_red = lda.fit_transform(scaler.fit_transform(X_full), y_full)
# for model_name, model in models.items():
#     clf = clone(model).fit(X_red, y_full)
#     key = f'lda_{model_name}'
#     trained_models[key] = clf
#     joblib.dump(clf, f'models/{key}.joblib')
#
# # Salvataggio risultati
#df_results.to_csv('/content/drive/MyDrive/Colab Notebooks/results/StratifiedGroup5Fold_separatedPCAfixedDim_allSpectrums1900.csv', index=False)
df_results.to_csv('results/5-StratifiedGroupByProtein5Fold_separatedPCAfixedDim_allSpectrums1900.csv', index=False)
# print(f"✅ Salvati {len(trained_models)} modelli in 'models/' e risultati in 'results/'")

In [ ]:
# # Caricamento ed esecuzione di un modello salvato
# MODEL_PATH = 'models'
# model_name = 'pca_RandomForest'  # Modello con miglior Recall
#
# # Carico il modello
# loaded_model = joblib.load(f'{MODEL_PATH}/{model_name}.joblib')
#
# # Esempio di predizione (richiede X_val preprocessato con la stessa pipeline)
# # y_pred = loaded_model.predict(X_val)
# # y_pred_proba = loaded_model.predict_proba(X_val)[:, 1]

# 7) Conclusioni

---

## Appendice A: Istruzioni per l'Esecuzione

### Esecuzione su Google Colab

```python
# 1. Monta Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Copia il dataset
!cp -r "/content/drive/MyDrive/dataset_bioactivities" "/content/"

# 3. Installa dipendenze (se necessario)
!pip install xgboost scikit-learn pandas matplotlib seaborn

# 4. Modifica BASE_PATH nel notebook
BASE_PATH = Path('/content/dataset_bioactivities/dataset_bioactivities')
```

### Esecuzione su HPC con GPU

```bash
# 1. Richiedi un nodo con GPU
#SBATCH --gres=gpu:1
#SBATCH --partition=gpu

# 2. Carica i moduli necessari
module load python/3.10
module load cuda/11.8

# 3. Attiva l'ambiente virtuale
source venv/bin/activate

# 4. Esegui il notebook
jupyter nbconvert --execute --to notebook main.ipynb
```

### Requisiti
- Python >= 3.8
- scikit-learn >= 1.0
- xgboost >= 1.5
- pandas >= 1.3
- numpy >= 1.21
- matplotlib >= 3.5
- seaborn >= 0.11